# MELA — inference

Applies the locked MELA ensemble to a file of patients and writes one CSV with, for every
patient, its ID, each model's probability and vote, the vote count, the ensemble prediction and
the MELA class. When the file carries a ground truth it also reports sensitivity, specificity
and exact confidence intervals, and adds `final_dgn` and `outcome` (TP / FN / TN / FP) to the CSV.

The imputer, the scaler and the three models are loaded as separate files and applied step by
step — **imputer first, then scaler**, as in training — with a check that the result matches the
saved pipelines. Each model votes positive when its probability is `>=` its locked threshold.

> Research code. MELA is not a certified medical device and must not drive patient care.

In [1]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.stats import beta
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

## 1) Configuration

Defaults point at the **synthetic** toy cohort so the notebook runs out of the box. Set
`DATA_PATH` to your own file (`.csv`, `.xlsx` or `.ods`) and `OUTPUT_PATH` to where the
predictions should go.

In [2]:
DATA_PATH = Path("../toy_data/toy_cohort.csv")     # <-- your cohort
MODELS_DIR = Path("../models")
OUTPUT_PATH = Path("../results/predictions/mela_predictions.csv")   # <-- output CSV

DROP_SHOCK = True      # exclude haemodynamically unstable patients, as in the study cohorts
ID_COLS = ("patient_id", "record_id", "Uniq_ID")   # first one found becomes the row index

## 2) Load imputer, scaler, models and thresholds

All produced by section 9 of `MELA_analysis.ipynb`. Each classifier is taken from its saved
pipeline; the imputer and scaler are the files saved on their own.

In [3]:
def load_artifact(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")
    return joblib.load(path)


with open(MODELS_DIR / "manifest.json") as fh:
    MANIFEST = json.load(fh)

MODEL_NAMES = tuple(MANIFEST["ensemble_members"])
N_VOTES_POSITIVE = MANIFEST["n_votes_positive"]
THRESHOLDS = MANIFEST["thresholds"]
FEATURES = MANIFEST["feature_order"]
FIXED_ZERO_COLS = MANIFEST["fixed_zero_columns"]
LOG2_COLS = MANIFEST["log2_transform"]

imputer = load_artifact(MODELS_DIR / "imputer_final.joblib")
scaler = load_artifact(MODELS_DIR / "scaler_final.joblib")
pipelines = {name: load_artifact(MODELS_DIR / f"pipeline_{name}.joblib") for name in MODEL_NAMES}
models = {name: pipe.named_steps["clf"] for name, pipe in pipelines.items()}

print("imputer :", type(imputer).__name__, f"({len(imputer.statistics_)} features)")
print("scaler  :", type(scaler).__name__)
for name in MODEL_NAMES:
    print(f"model   : {name:<9} {type(models[name]).__name__:<22} threshold = {THRESHOLDS[name]:.8g}")
print(f"\nEnsemble: positive if >= {N_VOTES_POSITIVE}/{len(MODEL_NAMES)} models vote positive")

imputer : SimpleImputer (28 features)
scaler  : StandardScaler
model   : XGBoost   XGBClassifier          threshold = 0.015033977
model   : AdaBoost  AdaBoostClassifier     threshold = 0.2867102
model   : CatBoost  CatBoostClassifier     threshold = 0.016142942

Ensemble: positive if >= 2/3 models vote positive


## 3) Load the cohort

Columns are matched by name. Anything not in the feature list is ignored, anything missing is
created empty and imputed later inside the pipeline. If the file has a patient ID column
(`patient_id`, `record_id` or `Uniq_ID`), rows are indexed by it and every output keeps it.

In [4]:
def load_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    elif suffix == ".ods":
        df = pd.read_excel(path, engine="odf")
    else:
        raise ValueError(f"unsupported format: {suffix}")
    print(f"Loaded {path.name}: {len(df)} rows, {df.shape[1]} columns")

    id_col = next((c for c in ID_COLS if c in df.columns), None)
    if id_col is not None:
        df = df.set_index(id_col)
        df.index.name = "patient_id"
        print(f"  patient ID column: {id_col}" + ("" if df.index.is_unique else "  (WARNING: not unique)"))
    else:
        df.index.name = "row"
        print("  no patient ID column: rows are numbered from 0")
    return df


df_raw = load_dataset(DATA_PATH)

missing = [c for c in FEATURES if c not in df_raw.columns]
extra = [c for c in df_raw.columns if c not in FEATURES and c not in {"final_dgn", "shock"}]
if missing:
    print(f"  absent, will be imputed : {missing}")
if extra:
    print(f"  ignored                 : {extra}")

Loaded toy_cohort.csv: 200 rows, 30 columns
  patient ID column: patient_id


## 4) Deterministic preprocessing

Three deterministic steps, none of them learned from the data:

- `smoke` harmonised from free text to 0/1;
- the four symptom variables coded as absent when missing;
- `ddimer_value` log2-transformed (supply it in ng/mL, as measured) and non-positive values
  dropped.

Mean imputation and standardisation come next, in section 5, with the statistics learned on the
development set.

In [5]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    mapping_smoke = {"never smoked": 0, "former smoker": 1, "unknown": 0,
                     "active smoker": 1, 0: 0, "0": 0}
    if "smoke" in df.columns and df["smoke"].dtype == object:
        df["smoke"] = df["smoke"].map(mapping_smoke).fillna(0).astype(int)

    for c in FIXED_ZERO_COLS:
        if c in df.columns:
            df[c] = df[c].fillna(0.0)

    if DROP_SHOCK and "shock" in df.columns:
        before = len(df)
        df = df[df["shock"] == 0].copy()
        print(f"  shock exclusion: {before - len(df)} patients removed")

    for c in LOG2_COLS:
        if c not in df.columns:
            raise ValueError(f"'{c}' is required and was not found")
        df.loc[df[c] <= 0, c] = np.nan
        before = len(df)
        df = df.dropna(subset=[c]).copy()
        if before != len(df):
            print(f"  {before - len(df)} patients dropped for missing/invalid {c}")
        df[c] = np.log2(df[c])

    return df


df_proc = preprocess(df_raw)
X = df_proc.reindex(columns=FEATURES)
print(f"\nReady to score: {X.shape[0]} patients x {X.shape[1]} features "
      f"({int(X.isna().sum().sum())} missing values left to the imputer)")


Ready to score: 200 patients x 28 features (21 missing values left to the imputer)


## 5) Imputer + scaler

Mean imputation with the development-set means, then standardisation with the development-set
mean and standard deviation. Same order as in training: **imputer first, then scaler**.

In [6]:
X_imputed = imputer.transform(X)
X_ready = scaler.transform(X_imputed)
print("Final shape:", X_ready.shape, "| missing after imputation:", int(np.isnan(X_ready).sum()))

# Safety check: the step-by-step result must equal what the saved pipelines compute.
for name in MODEL_NAMES:
    manual = models[name].predict_proba(X_ready)[:, 1]
    piped = pipelines[name].predict_proba(X)[:, 1]
    assert np.allclose(manual, piped), f"{name}: step-by-step preprocessing differs from the pipeline"
print("OK - step-by-step preprocessing matches the saved pipelines")

Final shape: (200, 28) | missing after imputation: 0
OK - step-by-step preprocessing matches the saved pipelines


## 6) Prediction (models + threshold + majority vote)

Each model votes positive when its probability is `>=` its locked threshold; the ensemble is
positive when at least 2 of the 3 models vote positive.

In [7]:
def predict_with_threshold(model, X, thr):
    proba = model.predict_proba(X)[:, 1]
    return proba, (proba >= thr).astype(int)


out = pd.DataFrame(index=df_proc.index)
votes = []
for name in MODEL_NAMES:
    proba, pred = predict_with_threshold(models[name], X_ready, THRESHOLDS[name])
    out[f"proba_{name}"] = proba
    out[f"pred_{name}"] = pred
    votes.append(pred)

out["n_positive_votes"] = np.vstack(votes).sum(axis=0)
positive = out["n_positive_votes"] >= N_VOTES_POSITIVE
out["prediction"] = positive.astype(int)
out["MELA_class"] = np.where(positive, "II (rule-in)", "I (rule-out)")

print(out["MELA_class"].value_counts().to_string())

MELA_class
I (rule-out)    110
II (rule-in)     90


## 7) Evaluation against a ground truth

Only runs when the file carries a label column. Confidence intervals are exact
Clopper–Pearson, the same as in the analysis.

On the toy cohort these numbers mean nothing: the data are invented.

In [8]:
def clopper_pearson(k: int, n: int, alpha: float = 0.05):
    if n <= 0:
        return np.nan, np.nan
    lo = 0.0 if k == 0 else beta.ppf(alpha / 2, k, n - k + 1)
    hi = 1.0 if k == n else beta.ppf(1 - alpha / 2, k + 1, n - k)
    return float(lo), float(hi)


label_col = next((c for c in ["final_dgn", "label", "y"] if c in df_proc.columns), None)

if label_col is None:
    print("No label column found - skipping evaluation.")
else:
    y_true = df_proc[label_col].astype(int).to_numpy()
    y_pred = out["prediction"].to_numpy()

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    s_lo, s_hi = clopper_pearson(tp, tp + fn)
    p_lo, p_hi = clopper_pearson(tn, tn + fp)

    print(f"Label column: {label_col}   n={len(y_true)}   positives={tp + fn}")
    print(f"  TP={tp}  FN={fn}  TN={tn}  FP={fp}")
    print(f"  Sensitivity {sens:.4f}  (95% CI {s_lo:.4f}-{s_hi:.4f})")
    print(f"  Specificity {spec:.4f}  (95% CI {p_lo:.4f}-{p_hi:.4f})")
    print()
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

Label column: final_dgn   n=200   positives=17
  TP=16  FN=1  TN=109  FP=74
  Sensitivity 0.9412  (95% CI 0.7131-0.9985)
  Specificity 0.5956  (95% CI 0.5207-0.6674)

              precision    recall  f1-score   support

           0     0.9909    0.5956    0.7440       183
           1     0.1778    0.9412    0.2991        17

    accuracy                         0.6250       200
   macro avg     0.5843    0.7684    0.5215       200
weighted avg     0.9218    0.6250    0.7062       200



## 8) Check against the analysis (optional)

When the cohort is one of those used in `MELA_analysis.ipynb` (for example the PRESC1SE file),
the probabilities computed here are compared, patient by patient, with the ones recorded by the
analysis. Skipped for any other cohort.

In [9]:
SHORT_NAME = {"XGBoost": "XGB", "AdaBoost": "ADA", "CatBoost": "CAT"}
reference_path = Path("../results/predictions/predictions_external_validation.csv")

if not reference_path.exists() or out.index.name != "patient_id":
    print("No reference available - skipping.")
else:
    ref = pd.read_csv(reference_path, index_col=0)
    common = out.index.intersection(ref.index)
    if len(common) == 0 or not (out.index.is_unique and ref.index.is_unique):
        print("Cohort is not the analysis external cohort - skipping.")
    else:
        for name in MODEL_NAMES:
            drift = np.abs(out.loc[common, f"proba_{name}"] - ref.loc[common, f"proba{SHORT_NAME[name]}"]).max()
            print(f"  {name:<10} max probability drift = {drift:.3e}")
        agree = int((out.loc[common, "prediction"] == ref.loc[common, "prediction"]).sum())
        print(f"  ensemble agreement: {agree}/{len(common)} patients in common")
        assert agree == len(common), "cohort inference does not reproduce the analysis"

Cohort is not the analysis external cohort - skipping.


## 9) Save the predictions

One row per patient, indexed by its ID: `proba_<MODEL>`, `pred_<MODEL>`, `n_positive_votes`,
`prediction` (1 = rule-in) and `MELA_class`; with a label column also `final_dgn` and `outcome`.

In [10]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
export = out.copy()
if label_col is not None:
    y = df_proc[label_col].astype(int).to_numpy()
    p = export["prediction"].to_numpy()
    export[label_col] = y
    export["outcome"] = np.select(
        [(y == 1) & (p == 1), (y == 1) & (p == 0), (y == 0) & (p == 0)], ["TP", "FN", "TN"], default="FP"
    )
export.to_csv(OUTPUT_PATH, index=True)
print("Saved:", OUTPUT_PATH, f"({len(export)} rows)")

Saved: ../results/predictions/mela_predictions.csv (200 rows)
